# Forward & Inverse Model — Port_S2 1-8GHz**Data:** l3=9~17 x w4=0.5~4.0 = 72 curves, 40 freq points (1.0-8.0 GHz)**Pipeline:**1. Convert .txt -> .xlsx (same format as Backfeed_Data300)2. Forward: (l3,w4) -> S11 via PCA+MLP, 5-Fold CV3. Inverse: S11 -> (l3,w4) via Tandem network, 5-Fold CV4. Visualizations & model export

In [ ]:
import copy, warnings, os, re, jsonimport matplotlib; matplotlib.use('TkAgg')import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport torchimport torch.nn as nnfrom sklearn.decomposition import PCAfrom sklearn.model_selection import KFoldfrom sklearn.preprocessing import StandardScalerwarnings.filterwarnings('ignore')matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']matplotlib.rcParams['axes.unicode_minus'] = FalseSEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {device}')

In [ ]:
# ========== 1. Convert to Excel ==========txt_path = '../Port_S_data_2参数_1-8ghz.txt'with open(txt_path) as f: header = f.readline().strip()matches = re.findall(r'S\(1,1\),l3=([\d.]+), w4=([\d.]+)', header)print(f'Curves: {len(matches)}, l3={sorted(set(float(m[0]) for m in matches))}')print(f'w4={sorted(set(float(m[1]) for m in matches))}')df = pd.read_csv(txt_path, sep=r'\s+', skiprows=[0,1,2,3], header=None, dtype=float, engine='python')freq_hz = df.iloc[:, 0].values; s11_raw = df.iloc[:, 1:].valuesrows = []for idx, (l3, w4) in enumerate(matches):    row = {'l3': float(l3), 'w4': float(w4), 'Unnamed: 15': np.nan}    for j, f in enumerate(freq_hz): row[f'{f/1e9:.6f}'] = s11_raw[j, idx]    rows.append(row)df_out = pd.DataFrame(rows)fcols = sorted([c for c in df_out.columns if c not in ['l3','w4','Unnamed: 15']], key=lambda x: float(x))df_out = df_out[['l3','w4','Unnamed: 15'] + fcols]df_out.to_excel('Port_S_data_2_1-8GHz.xlsx', index=False, engine='openpyxl')X = df_out[['l3', 'w4']].to_numpy(dtype=np.float32)y_db = df_out.iloc[:, 3:].to_numpy(dtype=np.float32)frequency = df_out.columns[3:].astype(float).to_numpy()print(f'Samples: {len(X)}, S11 points: {y_db.shape[1]}')print(f'Freq: {frequency[0]:.4f} ~ {frequency[-1]:.4f} GHz')

In [ ]:
# ========== 2. Forward Model ==========class ForwardNet(nn.Module):    def __init__(self, pca_dim):        super().__init__()        self.net = nn.Sequential(            nn.Linear(2, 128), nn.SiLU(),            nn.Linear(128, 128), nn.SiLU(),            nn.Linear(128, 128), nn.SiLU(),            nn.Linear(128, pca_dim),        )    def forward(self, x): return self.net(x)def wloss(pred, true):    w = 1 + 20 * (torch.clamp(-true, min=0) / 40) ** 2    return torch.mean(w * (pred - true) ** 2)print('ForwardNet defined.')

In [ ]:
# ========== 2a. 5-Fold CV ==========K = 5; kf = KFold(n_splits=K, shuffle=True, random_state=SEED)cv_va = []for fold, (tr_i, va_i) in enumerate(kf.split(X)):    x_sc = StandardScaler().fit(X[tr_i])    x_tr_s = x_sc.transform(X[tr_i]).astype(np.float32)    x_va_s = x_sc.transform(X[va_i]).astype(np.float32)    pdim = min(20, len(tr_i) - 2)    pca = PCA(n_components=pdim).fit(y_db[tr_i])    y_tr_c = pca.transform(y_db[tr_i]).astype(np.float32)    c_sc = StandardScaler().fit(y_tr_c)    y_tr_cs = c_sc.transform(y_tr_c).astype(np.float32)    Xtr = torch.tensor(x_tr_s, device=device); Xva = torch.tensor(x_va_s, device=device)    Ytr = torch.tensor(y_db[tr_i], device=device); Yva = torch.tensor(y_db[va_i], device=device)    Ytr_cs = torch.tensor(y_tr_cs, device=device)    pct = torch.tensor(pca.components_, dtype=torch.float32, device=device)    pmt = torch.tensor(pca.mean_, dtype=torch.float32, device=device)    cst = torch.tensor(c_sc.scale_, dtype=torch.float32, device=device)    cmt = torch.tensor(c_sc.mean_, dtype=torch.float32, device=device)    def dec(c): return (c * cst + cmt) @ pct + pmt    m = ForwardNet(pdim).to(device)    opt = torch.optim.AdamW(m.parameters(), lr=0.001, weight_decay=1e-5)    best_v, best_e, best_s, pat = float('inf'), 0, None, 0    for ep in range(5000):        m.train(); opt.zero_grad()        pc = m(Xtr); loss = wloss(dec(pc), Ytr) + 0.01 * torch.mean((pc - Ytr_cs) ** 2)        loss.backward(); opt.step()        m.eval()        with torch.no_grad():            v_db = dec(m(Xva)); vl = wloss(v_db, Yva)        if vl.item() < best_v - 1e-6:            best_v = vl.item(); best_e = ep + 1; best_s = copy.deepcopy(m.state_dict()); pat = 0        else: pat += 1        if pat >= 500: break    m.load_state_dict(best_s); m.eval()    with torch.no_grad():        p_va = dec(m(Xva)).cpu().numpy()    err = p_va - y_db[va_i]    tmi = np.argmin(y_db[va_i], axis=1); pmi = np.argmin(p_va, axis=1)    va_m = {'MAE': np.mean(np.abs(err)), 'FreqErr_MHz': np.mean(np.abs(frequency[pmi] - frequency[tmi])) * 1000}    cv_va.append(va_m)    print(f'Fold {fold+1}: Val MAE={va_m["MAE"]:.4f}, FreqErr={va_m["FreqErr_MHz"]:.1f} MHz')print(f'\nCV Mean MAE: {np.mean([m["MAE"] for m in cv_va]):.4f} +- {np.std([m["MAE"] for m in cv_va]):.4f}')

In [ ]:
# ========== 2b. Final Forward Model ==========x_sc_final = StandardScaler().fit(X)X_s_final = x_sc_final.transform(X).astype(np.float32)pca_final = PCA(n_components=25).fit(y_db)y_c_final = pca_final.transform(y_db).astype(np.float32)c_sc_final = StandardScaler().fit(y_c_final)y_cs_final = c_sc_final.transform(y_c_final).astype(np.float32)print(f'PCA var: {pca_final.explained_variance_ratio_.sum():.4%}')# Train with holdout ES (same as before)ho_n = max(5, len(X) // 10)ho_idx = np.random.RandomState(SEED).choice(len(X), size=ho_n, replace=False)tr_idx = np.setdiff1d(np.arange(len(X)), ho_idx)Xtr_f = torch.tensor(X_s_final[tr_idx], device=device)Xho_f = torch.tensor(X_s_final[ho_idx], device=device)Ytr_f = torch.tensor(y_db[tr_idx], device=device)Yho_f = torch.tensor(y_db[ho_idx], device=device)Ytr_cs_f = torch.tensor(y_cs_final[tr_idx], device=device)pct_f = torch.tensor(pca_final.components_, dtype=torch.float32, device=device)pmt_f = torch.tensor(pca_final.mean_, dtype=torch.float32, device=device)cst_f = torch.tensor(c_sc_final.scale_, dtype=torch.float32, device=device)cmt_f = torch.tensor(c_sc_final.mean_, dtype=torch.float32, device=device)def dec_final(c): return (c * cst_f + cmt_f) @ pct_f + pmt_fmodel_f = ForwardNet(25).to(device)opt_f = torch.optim.AdamW(model_f.parameters(), lr=0.001, weight_decay=1e-5)best_v, best_e, best_s, pat = float('inf'), 0, None, 0for ep in range(5000):    model_f.train(); opt_f.zero_grad()    pc = model_f(Xtr_f); loss = wloss(dec_final(pc), Ytr_f) + 0.01 * torch.mean((pc - Ytr_cs_f) ** 2)    loss.backward(); opt_f.step()    model_f.eval()    with torch.no_grad():        ho_db = dec_final(model_f(Xho_f)); vl = wloss(ho_db, Yho_f)    if vl.item() < best_v - 1e-6:        best_v = vl.item(); best_e = ep + 1; best_s = copy.deepcopy(model_f.state_dict()); pat = 0    else: pat += 1    if pat >= 500: breakmodel_f.load_state_dict(best_s); model_f.eval()with torch.no_grad():    X_all_t = torch.tensor(X_s_final, device=device)    pred_all = dec_final(model_f(X_all_t)).cpu().numpy()all_maes = np.mean(np.abs(pred_all - y_db), axis=1)print(f'Best epoch: {best_e}')print(f'MAE: mean={all_maes.mean():.4f}, median={np.median(all_maes):.4f}, max={all_maes.max():.4f}')# Savetorch.save({'model_state': model_f.state_dict(), 'param_cols': ['l3','w4'],    'x_mean': x_sc_final.mean_, 'x_scale': x_sc_final.scale_,    'coeff_mean': c_sc_final.mean_, 'coeff_scale': c_sc_final.scale_,    'pca_components': pca_final.components_, 'pca_mean': pca_final.mean_,    'frequency': frequency, 'input_dim': 2, 'pca_dim': 25}, 'model_forward.pth')print('Forward model saved: model_forward.pth')

In [ ]:
# ========== 3. Inverse Model (Tandem) ==========# Feature extractiondef extract_features(curves):    feats = []    for c in curves:        fi = np.argmin(c); fr = frequency[fi]; s11m = c[fi]        m10 = c <= -10; bw10 = frequency[m10][-1] - frequency[m10][0] if m10.sum() > 1 else 0.0        m3 = c <= -3; bw3 = frequency[m3][-1] - frequency[m3][0] if m3.sum() > 1 else 0.0        integral = np.trapezoid(c, frequency)        feats.append([fr, s11m, bw10, bw3, integral, np.mean(c), np.std(c)])    return np.array(feats, dtype=np.float32)pca_inv = PCA(n_components=25).fit(y_db)y_pca = pca_inv.transform(y_db).astype(np.float32)hand_feats = extract_features(y_db)feat_scaler = StandardScaler().fit(hand_feats)hand_feats_s = feat_scaler.transform(hand_feats).astype(np.float32)X_inv_raw = np.concatenate([y_pca, hand_feats_s], axis=1)inv_scaler = StandardScaler().fit(X_inv_raw)X_inv_s = inv_scaler.transform(X_inv_raw).astype(np.float32)y_params = X.copy()y_min, y_max = y_params.min(axis=0), y_params.max(axis=0)y_params_norm = (y_params - y_min) / (y_max - y_min)print(f'Inverse input dim: {X_inv_raw.shape[1]} (25 PCA + 7 features)')

In [ ]:
class InverseNet(nn.Module):    def __init__(self, in_dim, out_dim):        super().__init__()        self.net = nn.Sequential(            nn.Linear(in_dim, 256), nn.SiLU(), nn.Dropout(0.1),            nn.Linear(256, 256), nn.SiLU(), nn.Dropout(0.1),            nn.Linear(256, 128), nn.SiLU(),            nn.Linear(128, out_dim), nn.Sigmoid(),        )    def forward(self, x): return self.net(x)print('InverseNet defined.')

In [ ]:
# ========== 3a. Inverse 5-Fold CV ==========cv_inv_mae = []; cv_inv_curve = []for fold, (tr_i, va_i) in enumerate(kf.split(X)):    Xi_tr = torch.tensor(X_inv_s[tr_i], device=device)    Xi_va = torch.tensor(X_inv_s[va_i], device=device)    Yp_tr = torch.tensor(y_params_norm[tr_i], dtype=torch.float32, device=device)    Yp_va = torch.tensor(y_params_norm[va_i], dtype=torch.float32, device=device)    Ydb_tr = torch.tensor(y_db[tr_i], device=device)    Ydb_va = torch.tensor(y_db[va_i], device=device)    ym_t = torch.tensor(y_min, dtype=torch.float32, device=device)    yx_t = torch.tensor(y_max, dtype=torch.float32, device=device)    xm_t = torch.tensor(x_sc_final.mean_, dtype=torch.float32, device=device)    xs_t = torch.tensor(x_sc_final.scale_, dtype=torch.float32, device=device)    def tandem_loss(pred_norm, target_norm, target_db):        pl = torch.mean((pred_norm - target_norm) ** 2)        pd = pred_norm * (yx_t - ym_t) + ym_t        ps = (pd - xm_t) / xs_t        with torch.no_grad():            pc_f = dec_final(model_f(ps))        cl = wloss(pc_f, target_db)        return pl + 0.5 * cl    m_inv = InverseNet(X_inv_raw.shape[1], 2).to(device)    opt_i = torch.optim.AdamW(m_inv.parameters(), lr=0.0005, weight_decay=1e-4)    best_v, best_e, best_s, pat = float('inf'), 0, None, 0    for ep in range(5000):        m_inv.train(); opt_i.zero_grad()        pp = m_inv(Xi_tr); loss = tandem_loss(pp, Yp_tr, Ydb_tr)        loss.backward(); opt_i.step()        m_inv.eval()        with torch.no_grad():            pp_va = m_inv(Xi_va); vl = tandem_loss(pp_va, Yp_va, Ydb_va)        if vl.item() < best_v - 1e-6:            best_v = vl.item(); best_e = ep + 1; best_s = copy.deepcopy(m_inv.state_dict()); pat = 0        else: pat += 1        if pat >= 500: break    m_inv.load_state_dict(best_s); m_inv.eval()    with torch.no_grad():        pva_n = m_inv(Xi_va).cpu().numpy()        pva_d = pva_n * (y_max - y_min) + y_min        pva_s = x_sc_final.transform(pva_d.astype(np.float32))        pva_t = torch.tensor(pva_s, device=device)        rec_c = dec_final(model_f(pva_t)).cpu().numpy()    cv_inv_mae.append(np.mean(np.abs(pva_d - y_params[va_i])))    cv_inv_curve.append(np.mean(np.abs(rec_c - y_db[va_i])))    print(f'Fold {fold+1}: Param MAE={cv_inv_mae[-1]:.4f}, Curve MAE={cv_inv_curve[-1]:.4f}')print(f'\nCV Param MAE: {np.mean(cv_inv_mae):.4f} +- {np.std(cv_inv_mae):.4f}')print(f'CV Curve MAE: {np.mean(cv_inv_curve):.4f} +- {np.std(cv_inv_curve):.4f}')

In [ ]:
# ========== 3b. Final Inverse Model ==========Xi_all = torch.tensor(X_inv_s, device=device)Yp_all = torch.tensor(y_params_norm, dtype=torch.float32, device=device)Ydb_all = torch.tensor(y_db, device=device)Xi_tr_f = torch.tensor(X_inv_s[tr_idx], device=device)Xi_ho_f = torch.tensor(X_inv_s[ho_idx], device=device)Yp_tr_f = torch.tensor(y_params_norm[tr_idx], dtype=torch.float32, device=device)Yp_ho_f = torch.tensor(y_params_norm[ho_idx], dtype=torch.float32, device=device)Ydb_tr_f2 = torch.tensor(y_db[tr_idx], device=device)Ydb_ho_f = torch.tensor(y_db[ho_idx], device=device)ym_t2 = torch.tensor(y_min, dtype=torch.float32, device=device)yx_t2 = torch.tensor(y_max, dtype=torch.float32, device=device)xm_t2 = torch.tensor(x_sc_final.mean_, dtype=torch.float32, device=device)xs_t2 = torch.tensor(x_sc_final.scale_, dtype=torch.float32, device=device)def tl2(pn, tn, td):    pl = torch.mean((pn - tn) ** 2)    pd = pn * (yx_t2 - ym_t2) + ym_t2    ps = (pd - xm_t2) / xs_t2    with torch.no_grad(): pc_f = dec_final(model_f(ps))    return pl + 0.5 * wloss(pc_f, td)inv_final = InverseNet(X_inv_raw.shape[1], 2).to(device)opt_if = torch.optim.AdamW(inv_final.parameters(), lr=0.0005, weight_decay=1e-4)best_v, best_e, best_s, pat = float('inf'), 0, None, 0for ep in range(5000):    inv_final.train(); opt_if.zero_grad()    pp = inv_final(Xi_tr_f); loss = tl2(pp, Yp_tr_f, Ydb_tr_f2)    loss.backward(); opt_if.step()    inv_final.eval()    with torch.no_grad():        pho = inv_final(Xi_ho_f); vl = tl2(pho, Yp_ho_f, Ydb_ho_f)    if vl.item() < best_v - 1e-6:        best_v=vl.item(); best_e=ep+1; best_s=copy.deepcopy(inv_final.state_dict()); pat=0    else: pat+=1    if pat>=500: breakinv_final.load_state_dict(best_s); inv_final.eval()with torch.no_grad():    pall_n = inv_final(Xi_all).cpu().numpy()pall_d = pall_n * (y_max - y_min) + y_minparam_mae = np.mean(np.abs(pall_d - y_params), axis=0)print(f'Best epoch: {best_e}')print(f'l3 MAE: {param_mae[0]:.4f}, w4 MAE: {param_mae[1]:.4f}')# Savetorch.save({'model_state': inv_final.state_dict(), 'pca_inv': pca_inv,    'feat_scaler': feat_scaler, 'inv_scaler': inv_scaler,    'y_min': y_min, 'y_max': y_max, 'pca_dim_inv': 25, 'inp_dim': X_inv_raw.shape[1],    'frequency': frequency}, 'model_inverse.pth')print('Inverse model saved: model_inverse.pth')

In [ ]:
# ========== 4. Visualization Panel ==========# Reconstruct inverse predictionspall_s = x_sc_final.transform(pall_d.astype(np.float32))pall_t = torch.tensor(pall_s, device=device)with torch.no_grad(): rec_all = dec_final(model_f(pall_t)).cpu().numpy()rec_maes = np.mean(np.abs(rec_all - y_db), axis=1)fig, axes = plt.subplots(2, 3, figsize=(20, 12))# A: Forward CV boxplotax = axes[0, 0]cv_preds = np.zeros_like(y_db)for fi, (tri, vai) in enumerate(kf.split(X)):    xs_cv = StandardScaler().fit(X[tri]); xvs = xs_cv.transform(X[vai]).astype(np.float32)    pdi = min(20, len(tri) - 2); pcv = PCA(n_components=pdi).fit(y_db[tri])    ytcv = pcv.transform(y_db[tri]).astype(np.float32)    cscv = StandardScaler().fit(ytcv); ytcsv = cscv.transform(ytcv).astype(np.float32)    Xtcv = torch.tensor(xs_cv.transform(X[tri]).astype(np.float32), device=device)    Xvcv = torch.tensor(xvs, device=device)    Ytcv = torch.tensor(y_db[tri], device=device)    Ytcsv = torch.tensor(ytcsv, device=device)    pcvt = torch.tensor(pcv.components_, dtype=torch.float32, device=device)    pmvt = torch.tensor(pcv.mean_, dtype=torch.float32, device=device)    csct = torch.tensor(cscv.scale_, dtype=torch.float32, device=device)    cmct = torch.tensor(cscv.mean_, dtype=torch.float32, device=device)    def dcv(c): return (c * csct + cmct) @ pcvt + pmvt    mcv = ForwardNet(pdi).to(device); ocv = torch.optim.AdamW(mcv.parameters(), lr=0.001, weight_decay=1e-5)    for ep in range(1000):        mcv.train(); ocv.zero_grad()        pc = mcv(Xtcv); loss = wloss(dcv(pc), Ytcv) + 0.01 * torch.mean((pc - Ytcsv) ** 2)        loss.backward(); ocv.step()    mcv.eval()    with torch.no_grad(): cv_preds[vai] = dcv(mcv(Xvcv)).cpu().numpy()cv_data = [np.mean(np.abs(cv_preds[vai] - y_db[vai]), axis=1) for _, vai in kf.split(X)]ax.boxplot(cv_data, tick_labels=[f'Fold {i+1}' for i in range(K)])ax.set_ylabel('Curve MAE (dB)'); ax.set_title('Forward 5-Fold CV'); ax.grid(alpha=0.3, axis='y')# B: Inverse param scatterax = axes[0, 1]sc = ax.scatter(X[:,0], X[:,1], c=rec_maes, cmap='RdYlGn_r', s=50, edgecolors='gray', lw=0.5, label='True')ax.scatter(pall_d[:,0], pall_d[:,1], c=rec_maes, cmap='RdYlGn_r', s=50, marker='x', lw=1.5, label='Pred (inv)')for i in range(len(X)):    ax.plot([X[i,0], pall_d[i,0]], [X[i,1], pall_d[i,1]], '-', color='gray', alpha=0.2, lw=0.8)plt.colorbar(sc, ax=ax, label='Curve MAE (dB)')ax.set_xlabel('l3'); ax.set_ylabel('w4'); ax.set_title('Inverse: True vs Predicted'); ax.legend(fontsize=8)# C: Best inverse reconstructionax = axes[0, 2]bi = np.argmin(rec_maes)ax.plot(frequency, y_db[bi], '-', color='#2563EB', lw=2.5, label=f'Target (l3={X[bi,0]:.0f}, w4={X[bi,1]:.1f})')ax.plot(frequency, rec_all[bi], '--', color='#DC2626', lw=2, label=f'Reconstructed (MAE={rec_maes[bi]:.4f})')ax.set_xlabel('Freq (GHz)'); ax.set_ylabel('S11 (dB)')ax.set_title(f'Inverse Best | l3->{pall_d[bi,0]:.2f}, w4->{pall_d[bi,1]:.2f}'); ax.legend(fontsize=8); ax.grid(alpha=0.3)# D: Forward bestax = axes[1, 0]bf = np.argmin(all_maes)ax.plot(frequency, y_db[bf], '-', color='#2563EB', lw=2, label=f'True')ax.plot(frequency, pred_all[bf], '--', color='#DC2626', lw=2, label=f'Pred (MAE={all_maes[bf]:.4f})')ax.set_xlabel('Freq (GHz)'); ax.set_ylabel('S11 (dB)')ax.set_title(f'Forward Best | l3={X[bf,0]:.0f}, w4={X[bf,1]:.1f}'); ax.legend(fontsize=8); ax.grid(alpha=0.3)# E: Forward worstax = axes[1, 1]wf = np.argmax(all_maes)ax.plot(frequency, y_db[wf], '-', color='#2563EB', lw=2, label=f'True')ax.plot(frequency, pred_all[wf], '--', color='#DC2626', lw=2, label=f'Pred (MAE={all_maes[wf]:.4f})')ax.set_xlabel('Freq (GHz)'); ax.set_ylabel('S11 (dB)')ax.set_title(f'Forward Worst | l3={X[wf,0]:.0f}, w4={X[wf,1]:.1f}'); ax.legend(fontsize=8); ax.grid(alpha=0.3)# F: MAE comparisonax = axes[1, 2]bins = np.linspace(0, max(all_maes.max(), rec_maes.max()), 15)ax.hist(all_maes, bins=bins, alpha=0.6, color='steelblue', edgecolor='white', label=f'Fwd (mean={np.mean(all_maes):.3f})')ax.hist(rec_maes, bins=bins, alpha=0.6, color='darkorange', edgecolor='white', label=f'Inv (mean={np.mean(rec_maes):.3f})')ax.set_xlabel('Curve MAE (dB)'); ax.set_title('Forward vs Inverse MAE'); ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')plt.suptitle('Port_S2 1-8GHz — Forward & Inverse Results', fontsize=15, fontweight='bold', y=1.01)plt.tight_layout()plt.savefig('model_results_summary.png', dpi=150, bbox_inches='tight')plt.show()print('Done!')